In [ ]:
!pip install catboost

In [ ]:
import numpy
import pandas
from catboost import CatBoostClassifier

In [ ]:
def preprocess(df):

    data = df.sort_values(['session_id','index'])
    data['event_time'] = 0
    try:
        data.loc[1:,'event_time'] = (data.elapsed_time[1:].reset_index() - data.elapsed_time[:-1].reset_index())['elapsed_time'].tolist()
        data.loc[data['index']==0,'event_time']=0
    except:
        pass

    cutscene_click =  data[data.event_name=='cutscene_click'].groupby('session_id').mean()[['event_time']]
    person_click = data[data.event_name=='person_click'].groupby('session_id').agg({'event_time':['mean'],'fqid':['nunique','count']})
    navigate_click= data[data.event_name=='navigate_click'].groupby('session_id').agg({'event_time':'count','room_coor_x':['mean','std'], 'room_coor_y':['mean','std'],'screen_coor_x':['mean','std'], 'screen_coor_y':['mean','std']})
    observation_click = data[data.event_name=='observation_click'].groupby('session_id').agg({'fqid':['nunique','count']})
    notification_click = data[data.event_name=='notification_click'].groupby('session_id').agg({'room_fqid':['nunique','count']})
    object_click = data[data.event_name=='object_click'].groupby(['session_id']).agg({'event_time':['count']})
    object_hover = data[data.event_name == 'object_hover'].groupby('session_id').agg({'hover_duration':['mean','count']})
    map_hoover =  data[data.event_name == 'map_hover'].groupby('session_id').agg({'hover_duration':['mean','count']})
    map_click  = data[data.event_name == 'map_click'].groupby('session_id').agg({'room_fqid':['nunique','count']})

    cutscene_click.columns = ['screen_avg_time']
    person_click.columns = [ 'person_click' + '_'.join(i[1:]) for i in person_click.columns ]
    navigate_click.columns = [ 'navigate_click' + '_'.join(i) for i in navigate_click.columns ]
    observation_click.columns = ['observation_click' + '_'.join(i[1:]) for i in observation_click.columns]
    notification_click.columns = ['notification_click' +  '_'.join(i[1:]) for i in notification_click.columns ]
    object_click.columns = ['object_click']
    object_hover.columns = ['object_hover' +  '_'.join(i[1:]) for i in object_hover.columns ]
    map_hoover.columns = ['map_hoover' +  '_'.join(i[1:]) for i in map_hoover.columns ]
    map_click.columns = ['map_click' +  '_'.join(i[1:]) for i in map_click.columns ]

    return pandas.concat([cutscene_click, person_click, navigate_click, \
        observation_click, notification_click, object_click, \
            object_hover, map_click, map_hoover,],axis=1).fillna(0)

In [ ]:
models = [ CatBoostClassifier().load_model('/kaggle/input/catboost-model/model_'+str(i)) for i in range(1,19) ]

In [ ]:
threshold_models =[0.45,0.84,0.65,0.4 ,0.53,0.43,0.46,0.48,0.45,0.53,0.5 ,0.44,0.59,0.46,0.53,0.46,0.47,0.61]


In [ ]:
for i in range(18):
    models[i].set_probability_threshold(threshold_models[i])

In [ ]:
import jo_wilder_310
env = jo_wilder_310.make_env()
iter_test = env.iter_test()

In [ ]:
for (test, sample_submission) in iter_test:
    data_1 =  test[test.level_group == '0-4']
    data_2 =  test[test.level_group == '5-12']
    data_3 =  test[test.level_group == '13-22']
    
    if data_1.size!=0:
        temp = preprocess(data_1)
        for i in range(1,4):
            predict = models[i-1].predict(temp)
            for sess in range(temp.shape[0]):
                tmp = str(temp.index[sess])+'_q'+str(i)
                mask = sample_submission.session_id == tmp
                sample_submission.loc[mask,'correct'] = predict[sess]
                
    if data_2.size!=0:
        temp = preprocess(data_2)
        for i in range(4,14):
            predict = models[i-1].predict(temp)
            for sess in range(temp.shape[0]):
                tmp = str(temp.index[sess])+'_q'+str(i)
                mask = sample_submission.session_id == tmp
                sample_submission.loc[mask,'correct'] = predict[sess]
                
    if data_3.size!=0:
        temp = preprocess(data_3)
        for i in range(14,19):
            predict = models[i-1].predict(temp)
            for sess in range(temp.shape[0]):
                tmp = str(temp.index[sess])+'_q'+str(i)
                mask = sample_submission.session_id == tmp
                sample_submission.loc[mask,'correct'] = predict[sess]
                
                
                
    env.predict(sample_submission)

In [ ]:
pandas.read_csv('/kaggle/working/submission.csv')